## GRC question generation

Pipeline note:
- this notebook consumes the canonical semantic matching output `nis2_nist_semantic_matches.csv`
- Flan-T5 is used to draft audit questions from the matched NIST controls


In [ ]:
# 📥 Install openpyxl (if not already installed)
!pip install -q openpyxl

import ast
import pandas as pd

# 1. Load the canonical NIS2-NIST semantic matching results
df_match = pd.read_csv("nis2_nist_semantic_matches.csv")
df_match["Matched Controls"] = df_match["Matched Controls"].apply(ast.literal_eval)

# 2. Extract unique Control IDs that matched
relevant_ids = sorted({
    control_id
    for matches in df_match["Matched Controls"]
    for control_id, _ in matches
})

# 3. Save to disk for inspection
pd.Series(relevant_ids, name="Control ID").to_csv("matched_control_ids.csv", index=False)

print(f"✅ Extracted {len(relevant_ids)} relevant Control IDs to 'matched_control_ids.csv'.")


📌 2. Load original SP800-53 Excel and filter for relevant controls

In [ ]:
# 📌 2: Load the correct sheet from the SP800-53 Excel

import pandas as pd

# 1. Read the matched Control IDs
matched_ids = pd.read_csv("matched_control_ids.csv")["Control ID"].tolist()

# 2. Load the Excel using the actual sheet name
df_catalog = pd.read_excel(
    "sp800-53r5-control-catalog.xlsx",
    sheet_name="SP 800-53 Revision 5"
)

# 3. Filter to only the relevant controls
df_rel = df_catalog[df_catalog["Control Identifier"].isin(matched_ids)].copy()
print(f"✅ Found {len(df_rel)} matching controls in the original catalog")

# 4. Display a sample of the relevant controls
df_rel[[
    "Control Identifier",
    "Control (or Control Enhancement) Name",
    "Control Text"
]].head()

📌 3. Generate GRC-style questions using the Hugging Face QG model

In [ ]:
!pip install -q transformers torch openpyxl

In [ ]:
# 📌 3.5 Generate detailed GRC yes/no questions using Flan-T5

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import files

# 1. Device setup (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 2. Load relevant Control IDs from semantic matching
df_match = pd.read_csv("nis2_nist_semantic_matches.csv")
import ast
relevant_ids = {
    control_id
    for matches in df_match["Matched Controls"].apply(ast.literal_eval)
    for control_id, _ in matches
}

# 3. Load the original SP800-53 catalog
df_catalog = pd.read_excel(
    "sp800-53r5-control-catalog.xlsx",
    sheet_name="SP 800-53 Revision 5"
)
df_rel = df_catalog[df_catalog["Control Identifier"].isin(relevant_ids)].copy()
print(f"✅ {len(df_rel)} relevant controls loaded")

# 4. Initialize Flan-T5
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device).eval()
print(f"✅ Loaded {model_name} on {device}")

# 5. Question generation function
def generate_questions(row, max_len=128):
    cid = row["Control Identifier"]
    text = row["Control Text"].strip()
    prompt = (
        "You are a security auditor writing detailed technical yes/no verification questions. "
        "Break the following control text into its individual requirements (clauses or bullets), "
        "and for each one generate a clear, concise yes/no question that an Application Expert can answer to confirm implementation.\n\n"
        f"Control ID: {cid}\n"
        f"Text: {text}\n\n"
        "List each question on a new line, ending with a question mark.\n"
        "Questions:"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    outputs = model.generate(**inputs, max_length=max_len, num_beams=4, do_sample=False)
    out = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    return [q.strip() for q in out.split("\n") if q.strip()]

# 6. Generate all questions
records = []
for _, row in df_rel.iterrows():
    for q in generate_questions(row):
        records.append({
            "Control ID": row["Control Identifier"],
            "Question": q
        })

# 7. Save and download
questions_df = pd.DataFrame(records)
output_file = "nist_controls_questions_detailed_flan.csv"
questions_df.to_csv(output_file, index=False)
print(f"✅ Generated {len(records)} questions into '{output_file}'")
files.download(output_file)


In [ ]:
# 📌 3.5 Generate detailed GRC yes/no/Not Applicable questions using Flan-T5 (refined prompt)

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import files

# 1. Device setup (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 2. Load matched Control IDs from semantic matching
df_match = pd.read_csv("nis2_nist_semantic_matches.csv")
import ast
relevant_ids = {
    control_id
    for matches in df_match["Matched Controls"].apply(ast.literal_eval)
    for control_id, _ in matches
}

# 3. Load original SP800-53 catalog and filter relevant controls
df_catalog = pd.read_excel(
    "sp800-53r5-control-catalog.xlsx",
    sheet_name="SP 800-53 Revision 5"
)
df_rel = df_catalog[df_catalog["Control Identifier"].isin(relevant_ids)].copy()
print(f"✅ {len(df_rel)} relevant controls loaded")

# 4. Initialize Flan-T5
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device).eval()
print(f"✅ Loaded {model_name} on {device}")

# 5. Refined question-generation function
def generate_questions(row, max_len=128):
    cid = row["Control Identifier"]
    text = row["Control Text"].strip()
    prompt = (
        "You are a security auditor writing technical yes/no/Not Applicable questions for a GRC questionnaire. "
        "For each distinct requirement clause in the control text, generate a single concise verification question "
        "that an Application Expert could answer to confirm implementation. "
        "Each question must:\n"
        "- Start with "Has the organization" or "Is the organization"\n"
        "- Be written in a formal, precise, process-oriented tone suitable for regulated environments\n"
        "- Accurately reflect exactly one requirement from the control text\n"
        "- End with a question mark\n"
        "- Allow responses: Yes / No / Not Applicable\n\n"
        f"Control ID: {cid}\n"
        f"Control Text: {text}\n\n"
        "Questions:"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    outputs = model.generate(
        **inputs,
        max_length=max_len,
        num_beams=4,
        no_repeat_ngram_size=2,
        early_stopping=True
    )
    out = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    lines = [line.strip() for line in out.split("\n") if line.strip()]
    return [line if line.endswith("?") else line + "?" for line in lines]

# 6. Generate all questions
records = []
for _, row in df_rel.iterrows():
    for q in generate_questions(row):
        records.append({
            "Control ID": row["Control Identifier"],
            "Question": q
        })

# 7. Save and download
questions_df = pd.DataFrame(records)
output_file = "nist_controls_questions_refined_v2.csv"
questions_df.to_csv(output_file, index=False)
print(f"✅ Generated {len(records)} questions into '{output_file}'")
files.download(output_file)
